In [0]:
dbutils.widgets.text("catalog_name",    "food_inspection")
dbutils.widgets.text("raw_schema",      "raw_zone")
dbutils.widgets.text("bronze_schema",   "bronze")
dbutils.widgets.text("metadata_schema", "metadata")
 
catalog_name    = dbutils.widgets.get("catalog_name")
raw_schema      = dbutils.widgets.get("raw_schema")
bronze_schema   = dbutils.widgets.get("bronze_schema")
metadata_schema = dbutils.widgets.get("metadata_schema")
 
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {bronze_schema}")
 
print(f"Source: {catalog_name}.{raw_schema}.chicago")
print(f"Target: {catalog_name}.{bronze_schema}.bronze_chicago")

In [0]:
%run ./_common_helpers

In [0]:
# STEP 1: Read Raw Chicago Delta table
 
from datetime import datetime
from pyspark.sql.functions import lit, current_timestamp
 
df = spark.read.table(f"{catalog_name}.{raw_schema}.chicago")
 
source_count = df.count()
print(f"Read {source_count} rows from {raw_schema}.chicago")
print(f"Columns: {len(df.columns)}")
 

In [0]:
#STEP 2: Add audit columns
# No transformations on actual data — just lineage metadata
 
df = (df
    .withColumn("_source_city", lit("Chicago"))
    .withColumn("_bronze_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit("Chicago.csv"))
)
 
print(f"Audit columns added: _source_city, _ingestion_timestamp, _source_file")
print(f"Total columns now: {len(df.columns)}")

In [0]:
# STEP 3: Write to Bronze Delta table
 
bronze_table = f"{catalog_name}.{bronze_schema}.bronze_chicago"
 
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(bronze_table)
 
target_count = spark.read.table(bronze_table).count()
print(f"Written {target_count} rows to {bronze_table}")

In [0]:
# STEP 4: Validate row counts
 
if source_count == target_count:
    status = "SUCCESS"
    print(f"Row count validated: {source_count} rows match")
else:
    status = "FAILED"
    print(f"Mismatch — source: {source_count} | target: {target_count}")

In [0]:
# STEP 5: Log to execution_log
 
from pyspark.sql import Row
 
log_row = spark.createDataFrame([Row(
    table_name       = "bronze_chicago",
    city             = "Chicago",
    execution_time   = datetime.now(),
    status           = status,
    source_row_count = source_count,
    target_row_count = target_count,
    file_location    = f"{catalog_name}.{raw_schema}.chicago",
    created_date     = datetime.now().date()
)])
 
log_row.write.format("delta") \
    .mode("append") \
    .saveAsTable(f"{catalog_name}.{metadata_schema}.execution_log")
 
print(f"Logged to execution_log. Status: {status}")

In [0]:
# STEP 6: Verify
 
print("Bronze Chicago table preview:")
display(spark.sql(f"SELECT * FROM {catalog_name}.{bronze_schema}.bronze_chicago LIMIT 5"))
 
# COMMAND ----------
 
print("Execution log:")
display(spark.sql(f"SELECT * FROM {catalog_name}.{metadata_schema}.execution_log"))

In [0]:
# STEP 7: Fail task if validation failed
# Uses _common_helpers.rollback_table() to roll back to the *previous* Delta version
# (not VERSION 0 which would wipe all prior history).

if status == "FAILED":
    rollback_table(f"{catalog_name}.{bronze_schema}.bronze_chicago")
    raise Exception("Row count mismatch for Bronze Chicago. Check execution_log.")
